what the output consist of for the first run: caffeinate -i uv run python src/cnn/train_vgg.py --device mps --epochs 50 --out-dir results/runs/ref_baseline;  

In [1]:
import numpy as np
z = np.load("results/runs/ref_baseline/projections.npz")
for k in z.files: print(k, z[k].shape, z[k].dtype)

zero_fraction (390, 2) float64
layer/features.0.weight (390, 10) float64
layer/classifier.weight (390, 10) float64


PyTorch file written with torch.save. It holds the model’s state_dict, a dictionary from parameter names to tensors: the weights at the end of training, nothing else. There is no optimizer state and no training history

In [2]:
import torch
sd = torch.load("results/runs/ref_baseline/state_dict.pt", map_location="cpu")
for k, v in sd.items(): print(k, tuple(v.shape))

features.0.weight (512, 3, 3, 3)
features.0.bias (512,)
classifier.weight (10, 131072)
classifier.bias (10,)


In [3]:
import numpy as np
z = np.load("results/runs/ref_baseline/projections.npz")
ms, N = (2, 3, 6, 9, 18, 36), 360

def alpha_m(m, x):                       # x: 1-D series whose length is divisible by m
    y = x.reshape(-1, m).sum(1); eps = np.spacing(1)
    return 1 / ((np.log(np.abs(y) + eps).mean() - np.log(np.abs(x) + eps).mean()) / np.log(m))

def layer_alpha(series, kind):           # series: (steps, 10 projections)
    out = []
    for col in series.T:
        x = col[-(N + 1):]
        x = np.diff(x) if kind == "increments" else x[1:]
        x = x - x.mean()
        out.append(np.median([alpha_m(m, x) for m in ms]))
    return np.median(out)

for key in [k for k in z.files if k.startswith("layer/")]:
    print(key, {kind: round(float(layer_alpha(z[key], kind)), 2) for kind in ("iterates", "increments")})
print("mean zero fraction per layer:", z["zero_fraction"].mean(0))

layer/features.0.weight {'iterates': 1.06, 'increments': 2.86}
layer/classifier.weight {'iterates': 0.99, 'increments': 2.96}
mean zero fraction per layer: [3.37577160e-05 1.26667512e-05]
